# order_reviews の重複調査

## 背景

`src/check_relationships.py` の検証で、以下が見つかった。

- `order_reviews(order_id) -> orders(order_id)` は 1:1 を想定していたが、実際は 1:N (最大 3 件)
- `review_id` 単独では一意にならない (重複 814 件)。主キーは `(review_id, order_id)`

## 目的

1 注文に複数のレビューが付いているケースについて、実際にどんな行が入っているかを確認し、
後続の整形で「どのレビューを採用するか」を決める材料にする。

In [1]:
from pathlib import Path

import pandas as pd

# notebook はこのファイルのあるディレクトリ (notebooks/) をカレントとして実行される想定
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
reviews.shape

(99224, 7)

## 1. 規模

`order_id` が 2 行以上出現する注文を抽出し、件数を確認する。

In [2]:
multi = (
    reviews[reviews.duplicated("order_id", keep=False)]
    .sort_values(["order_id", "review_creation_date"])
)
print(f"rows: {len(multi):,} / orders: {multi['order_id'].nunique():,}")

# 注文あたりのレビュー件数の分布
multi.groupby("order_id").size().value_counts().rename_axis("reviews_per_order").rename("orders")

rows: 1,098 / orders: 547


reviews_per_order
2    543
3      4
Name: orders, dtype: int64

## 2. 実際の行を見る

先頭のいくつかの注文について、レビュー行を並べて比較する。

In [3]:
sample_orders = multi["order_id"].drop_duplicates().head(6)
multi[multi["order_id"].isin(sample_orders)].drop(columns="review_comment_title")

,review_id,order_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,"Estou acostumada a comprar produtos pelo baratheon. Super confiável, transpa...",2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PEDIDO NAO RECEBI AINDA E OUTRO ...,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,ja reclamei varias vezes e ate hoje não sei onde foi parar o produto que com...,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


## 3. 同じ注文のレビュー同士はどれくらい似ているか

完全なコピーなのか、別々の回答なのかを確認する。

In [4]:
by_order = multi.groupby("order_id")

pd.Series(
    {
        "orders": multi["order_id"].nunique(),
        "same_review_id": (by_order["review_id"].nunique() == 1).sum(),
        "same_score": (by_order["review_score"].nunique() == 1).sum(),
        "same_creation_date": (by_order["review_creation_date"].nunique() == 1).sum(),
        # コメント無し (NaN) 同士も「同じ」とみなす
        "same_message": (by_order["review_comment_message"].nunique(dropna=False) == 1).sum(),
    }
)

orders                547
same_review_id          0
same_score            345
same_creation_date    155
same_message          317
dtype: int64

## 4. その review_id は他の注文にも付いているか

`review_id` の重複 (814 件) との関係を確認する。

In [5]:
shared_review_ids = reviews.loc[reviews.duplicated("review_id", keep=False), "review_id"]
n_orders = multi.loc[multi["review_id"].isin(shared_review_ids), "order_id"].nunique()
print(f"他の注文と review_id を共有しているレビューを持つ注文: {n_orders:,} / {multi['order_id'].nunique():,}")

他の注文と review_id を共有しているレビューを持つ注文: 458 / 547


## 考察 (暫定)

- 完全に同じ行のコピーではない。`review_id` はすべて異なり、スコアや作成日が異なるものも多い。
- 内容からは次の 2 パターンがありそう。
  - **日を空けた再回答**: 例 `02355020…` は配送トラブルが続き、スコアが 3 → 1 に下がっている。
  - **ほぼ同時の二重回答**: 例 `0176a684…` は回答時刻が数十秒差で、内容も同じ。
- 大半の注文 (458 / 547) で、そのレビューの `review_id` が別の注文にも付いている。
  1 つのレビューが複数の注文に紐づいている可能性が高い。

## 次に調べること

- `review_id` が複数の注文に付いているケースの中身。
  同じ顧客 (`customer_unique_id`) が同時期に出した複数注文に、1 件のレビューが紐づいているのではないか。